# [Kaggle] E2E BiGRU-CRF Baseline (Unified Tags)

Baseline model: **BiGRU-CRF** voi **Unified Tags** (B-CAMERA#POSITIVE, ...)

**Evaluation** (giong phobert-crf-absa.ipynb):
- Span-Level Exact Match F1
- Sentence-Level Multi-Label F1 (Micro/Macro/Weighted + per-label)


## 0. Setup


In [1]:
import subprocess, sys, os

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'underthesea', 'pytorch-crf', 'gensim'])

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/danghoang1302/uit-visd4sa'
    SAVE_DIR = '/kaggle/working/results/e2e_baseline'
    SRC_INPUT = '/kaggle/input/datasets/danghoang1302/absa-src'
    os.system(f'cp -r {SRC_INPUT}/src /kaggle/working/src')
    sys.path.insert(0, '/kaggle/working')
    print(f"KAGGLE | Data: {DATA_DIR}")
else:
    sys.path.insert(0, os.path.abspath(".."))
    DATA_DIR = os.path.join("..", "..", "data")
    SAVE_DIR = os.path.join("..", "..", "results", "e2e_baseline")
    print(f"LOCAL mode")

os.makedirs(SAVE_DIR, exist_ok=True)
for fn in ['train.jsonl', 'dev.jsonl', 'test.jsonl']:
    assert os.path.exists(os.path.join(DATA_DIR, fn)), f"MISSING: {fn}"
print("All data files OK!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.8 MB/s eta 0:00:00
KAGGLE | Data: /kaggle/input/datasets/danghoang1302/uit-visd4sa
All data files OK!


## 1. Data & Vocabulary


In [2]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import (load_raw_data, segment_items, build_vocab,
                                  train_w2v_embeddings)
from src.e2e.e2e_baseline_dataset import (E2EBaselineDataset,
                                           BIO_TAGS as UNIFIED_BIO_TAGS,
                                           NUM_TAGS as UNIFIED_NUM_TAGS,
                                           LABEL_NAMES)
from src.ate.ate_model import build_ate_model
from src.utils.metrics import (bio_tags_to_spans, evaluate_spans_f1,
                               bio_to_sentence_labels, evaluate_multilabel)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "train.jsonl")))
dev_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "dev.jsonl")))
test_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "test.jsonl")))

all_texts = [item["text"] for item in train_items + dev_items + test_items]
word2idx = build_vocab(all_texts, min_freq=2)
VOCAB_SIZE = len(word2idx)
EMB_DIM = 150
emb_matrix = train_w2v_embeddings(all_texts, word2idx, emb_dim=EMB_DIM)

print(f"Vocab: {VOCAB_SIZE} | Emb: {EMB_DIM}d | Unified Tags: {UNIFIED_NUM_TAGS}")
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")


Device: cuda
Training Word2Vec 150d...
Embedding Matrix ready. Hit: 6166/6168 (100.0%)
Vocab: 6168 | Emb: 150d | Unified Tags: 61
Train: 7785 | Dev: 1112 | Test: 2225


## 2. Datasets


In [3]:
MAX_LEN = 128
BATCH_SIZE = 64

train_ds = E2EBaselineDataset(train_items, word2idx, MAX_LEN)
dev_ds   = E2EBaselineDataset(dev_items,   word2idx, MAX_LEN)
test_ds  = E2EBaselineDataset(test_items,  word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_ds,   BATCH_SIZE)
test_loader  = DataLoader(test_ds,  BATCH_SIZE)
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")


Train: 7785 | Dev: 1112 | Test: 2225


## 3. Train


In [4]:
from src.utils.engine import train_ate_model, predict_ate

CONFIGS = [
    {"name": "BiGRU-CRF",  "type": "BiGRU",  "hidden": 256, "layers": 2, "lr": 1e-3},
    {"name": "BiLSTM-CRF", "type": "BiLSTM", "hidden": 256, "layers": 2, "lr": 1e-3},
]

results_list = []
all_mt = {}       # sentence-level per model
all_span = {}     # span-level per model
best_f1_global = 0
best_model_name = ""
best_test_res = None

for cfg in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  E2E {cfg['name']} | Unified Tags: {UNIFIED_NUM_TAGS}")
    print(f"{'='*60}")

    model = build_ate_model(
        model_type=cfg['type'], vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
        hidden_dim=cfg['hidden'], num_tags=UNIFIED_NUM_TAGS,
        pretrained_emb=emb_matrix, n_layers=cfg['layers'], dropout=0.3
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,}")

    model, history = train_ate_model(
        model, train_loader, dev_loader, device,
        lr=cfg['lr'], epochs=30, patience=7,
        model_name=f"E2E-{cfg['name']}",
        bio_tags_list=UNIFIED_BIO_TAGS
    )

    # === Evaluate ===
    test_res = predict_ate(model, test_loader, device, bio_tags_list=UNIFIED_BIO_TAGS)

    # Span-Level
    pred_sp = [bio_tags_to_spans(pt, UNIFIED_BIO_TAGS, l)
               for pt, l in zip(test_res['pred_tags'], test_res['lengths'])]
    true_sp = [bio_tags_to_spans(tt, UNIFIED_BIO_TAGS, l)
               for tt, l in zip(test_res['true_tags'], test_res['lengths'])]
    span_res = evaluate_spans_f1(pred_sp, true_sp)
    all_span[cfg['name']] = span_res

    # Sentence-Level
    pred_sent, _ = bio_to_sentence_labels(
        test_res['pred_tags'], test_res['lengths'], UNIFIED_BIO_TAGS, LABEL_NAMES)
    true_sent, _ = bio_to_sentence_labels(
        test_res['true_tags'], test_res['lengths'], UNIFIED_BIO_TAGS, LABEL_NAMES)
    mt = evaluate_multilabel(true_sent, pred_sent, LABEL_NAMES)
    all_mt[cfg['name']] = mt

    print(f"  TEST | Span-F1: {span_res['f1']:.4f} | Micro-F1: {mt['micro']['f1']:.4f} | Macro-F1: {mt['macro']['f1']:.4f}")
    results_list.append({'Model': f"E2E-{cfg['name']}",
        'Span_P': span_res['precision'], 'Span_R': span_res['recall'], 'Span_F1': span_res['f1'],
        'Micro_F1': mt['micro']['f1'], 'Macro_F1': mt['macro']['f1'], 'Params': n_params})

    if span_res['f1'] > best_f1_global:
        best_f1_global = span_res['f1']
        best_model_name = cfg['name']
        best_test_res = test_res
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"best_e2e_baseline_{cfg['name']}.pt"))
        print(f"  -> Saved as best!")



  E2E BiGRU-CRF | Unified Tags: 61
  Params: 2,885,456

  Training E2E-BiGRU-CRF


Training:   0%|          | 0/122 [00:00<?, ?it/s]

  Ep  1/30 | Train Loss: 78.6458 | Dev Loss: 52.1670 | TokAcc: 0.5461 | SpanF1: 0.1052 | LR: 0.001000 | 20.6s ***


Training:   0%|          | 0/122 [00:00<?, ?it/s]

  Ep  2/30 | Train Loss: 47.7552 | Dev Loss: 40.5804 | TokAcc: 0.6054 | SpanF1: 0.1799 | LR: 0.001000 | 19.4s ***


...

  Ep 27/30 | Train Loss: 4.6897 | Dev Loss: 26.5491 | TokAcc: 0.6538 | SpanF1: 0.3689 | LR: 0.000250 | 19.6s


Training:   0%|          | 0/122 [00:00<?, ?it/s]

  Ep 28/30 | Train Loss: 4.4928 | Dev Loss: 26.7289 | TokAcc: 0.6576 | SpanF1: 0.3696 | LR: 0.000250 | 19.6s


Training:   0%|          | 0/122 [00:00<?, ?it/s]

  Ep 29/30 | Train Loss: 4.3443 | Dev Loss: 27.5586 | TokAcc: 0.6595 | SpanF1: 0.3696 | LR: 0.000250 | 19.6s
  Early stopping at epoch 29 (Best Span F1: 0.3728)
  TEST | Span-F1: 0.3634 | Micro-F1: 0.7918 | Macro-F1: 0.5871
  -> Saved as best!


## 4. Chi tiet tung model


In [5]:
for model_name in all_mt.keys():
    mt = all_mt[model_name]
    sr = all_span[model_name]

    print(f"\n{'='*60}")
    print(f"  E2E-{model_name} — Full Evaluation")
    print(f"{'='*60}")

    # --- Span-Level ---
    print(f"\n  [A] SPAN-LEVEL (Exact Match)")
    print(f"      Precision : {sr['precision']:.4f}")
    print(f"      Recall    : {sr['recall']:.4f}")
    print(f"      F1        : {sr['f1']:.4f}")
    print(f"      TP={sr['tp']}  FP={sr['fp']}  FN={sr['fn']}")

    # --- Sentence-Level ---
    print(f"\n  [B] SENTENCE-LEVEL (Multi-Label)")
    print(f"      {'Average':<12} {'P':>8} {'R':>8} {'F1':>8}")
    print(f"      {'-'*40}")
    for avg in ['micro', 'macro', 'weighted']:
        m = mt[avg]
        print(f"      {avg:<12} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f1']:>8.4f}")

    print(f"\n      {'Label':<25} {'P':>7} {'R':>7} {'F1':>7} {'Sup':>6}")
    print(f"      {'-'*55}")
    for ln in LABEL_NAMES:
        m = mt[ln]
        flag = ' !' if m['support'] < 20 else ''
        print(f"      {ln:<25} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f} {m['support']:>6d}{flag}")



  E2E-BiGRU-CRF — Full Evaluation

  [A] SPAN-LEVEL (Exact Match)
      Precision : 0.3657
      Recall    : 0.3427
      F1        : 0.3538
      TP=2394  FP=4152  FN=4592

  [B] SENTENCE-LEVEL (Multi-Label)
      Average             P        R       F1
      ----------------------------------------
      micro          0.8166   0.7710   0.7932
      macro          0.6431   0.5623   0.5915
      weighted       0.8129   0.7710   0.7894

      Label                           P       R      F1    Sup
      -------------------------------------------------------
      CAMERA#POSITIVE            0.8817  0.8663  0.8739    344
      CAMERA#NEUTRAL             0.6415  0.4533  0.5312     75
      CAMERA#NEGATIVE            0.7405  0.7874  0.7632    174
      FEATURES#POSITIVE          0.8241  0.7417  0.7807    240
      FEATURES#NEUTRAL           0.4545  0.3226  0.3774     31
      FEATURES#NEGATIVE          0.8506  0.8037  0.8265    489
      PERFORMANCE#POSITIVE       0.8772  0.8772  0.8772

## 5. Summary & Comparison


In [6]:
results_df = pd.DataFrame(results_list).sort_values('Span_F1', ascending=False)
print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
display(results_df)
results_df.to_csv(os.path.join(SAVE_DIR, "e2e_baseline_results.csv"), index=False)

print(f"\n{'='*60}")
print(f"  SO SANH VOI phobert-crf-absa.ipynb baselines")
print(f"{'='*60}")
baseline_results = {
    'TextCNN-CRF': 0.7858, 'RNN-CRF': 0.7614, 'LSTM-CRF': 0.7622,
    'BiLSTM-CRF': 0.7853, 'GRU-CRF': 0.7639, 'BiGRU-CRF': 0.7966}
print(f"  {'Model':<25} {'Micro F1':>10}")
print(f"  {'-'*40}")
for n, f in baseline_results.items():
    print(f"  {n:<25} {f:>10.4f}")
print(f"  {'-'*40}")
for _, row in results_df.iterrows():
    print(f"  {row['Model']:<25} {row['Micro_F1']:>10.4f}  <- NB05")



  SUMMARY


,Model,Span_P,Span_R,Span_F1,Micro_F1,Macro_F1,Params
1,E2E-BiLSTM-CRF,0.378903,0.349127,0.363406,0.791845,0.587067,3488592
0,E2E-BiGRU-CRF,0.365720,0.342685,0.353828,0.793164,0.591475,2885456



  SO SANH VOI phobert-crf-absa.ipynb baselines
  Model                       Micro F1
  ----------------------------------------
  TextCNN-CRF                   0.7858
  RNN-CRF                       0.7614
  LSTM-CRF                      0.7622
  BiLSTM-CRF                    0.7853
  GRU-CRF                       0.7639
  BiGRU-CRF                     0.7966
  ----------------------------------------
  E2E-BiLSTM-CRF                0.7918  <- NB05
  E2E-BiGRU-CRF                 0.7932  <- NB05


## 6. Demo: Test Predictions


In [7]:
import random
random.seed(42)
n_samples = min(10, len(test_items))
sample_indices = random.sample(range(len(test_items)), n_samples)

print(f"{'='*70}")
print(f"  E2E-{best_model_name} DEMO: {n_samples} cau test")
print(f"{'='*70}")

demo_tp, demo_fp, demo_fn = 0, 0, 0
for idx in sample_indices:
    text = test_items[idx]['text']
    words = text.split()[:MAX_LEN]
    pred_tags = best_test_res['pred_tags'][idx]
    true_tags = best_test_res['true_tags'][idx]
    length = best_test_res['lengths'][idx]

    pred_spans = bio_tags_to_spans(pred_tags, UNIFIED_BIO_TAGS, length)
    true_spans = bio_tags_to_spans(true_tags, UNIFIED_BIO_TAGS, length)
    true_set, pred_set = set(true_spans), set(pred_spans)
    demo_tp += len(true_set & pred_set)
    demo_fp += len(pred_set - true_set)
    demo_fn += len(true_set - pred_set)

    print(f"\n{'─'*70}")
    print(f"  [{idx}] {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"  TRUE ({len(true_spans)}):")
    for label, s, e in true_spans:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in pred_set else 'MISSED'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    print(f"  PRED ({len(pred_spans)}):")
    for label, s, e in pred_spans:
        aspect = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = 'OK' if (label, s, e) in true_set else 'WRONG'
        print(f"    {label:<30} [{s}:{e}] \"{aspect}\"  {match}")
    if not pred_spans:
        print(f"    (khong co prediction)")

print(f"\n{'='*70}")
print(f"  Demo: TP={demo_tp}, FP={demo_fp}, FN={demo_fn}")


  E2E-BiLSTM-CRF DEMO: 10 cau test

──────────────────────────────────────────────────────────────────────
  [456] sản_phẩm sài tốt game mượt , hoàn_hảo chưa có vấn_đề gì . nhân_viên phục_vụ rất tốt_👍 👍_😄 😄
  TRUE (4):
    GENERAL#POSITIVE               [0:3] "sản_phẩm sài tốt"  OK
    PERFORMANCE#POSITIVE           [3:5] "game mượt"  OK
    GENERAL#POSITIVE               [6:11] "hoàn_hảo chưa có vấn_đề gì"  OK
    SER&ACC#POSITIVE               [11:16] ". nhân_viên phục_vụ rất tốt_👍"  MISSED
  PRED (4):
    GENERAL#POSITIVE               [0:3] "sản_phẩm sài tốt"  OK
    PERFORMANCE#POSITIVE           [3:5] "game mượt"  OK
    GENERAL#POSITIVE               [6:11] "hoàn_hảo chưa có vấn_đề gì"  OK
    SER&ACC#POSITIVE               [11:17] ". nhân_viên phục_vụ rất tốt_👍 👍_😄"  WRONG

──────────────────────────────────────────────────────────────────────
  [102] sau hơn một tháng sử_dụng mình đánh_giá tất_cả mọi thứ ok_nha , không biết sao nhiều người nói nhiều...
  TRUE (1):
    GENERAL#